# 06 · Urban growth 2016–2023 and heat

The challenge asks us to *quantify urban growth and map land-use transitions*. Google Open Buildings 2.5D Temporal (already our source of building heights) has one building map per year, 2016–2023, predicted from Sentinel-2 at an effective 4 m. Its **building presence** band is a model confidence (0–1).

Steps:
1. Per year, a 4 m pixel is built when presence ≥ `growth.presence_min` (0.5); built pixels are averaged to the same 90 m blocks as the heat model.
2. Blocks are classed *built before 2016*, *built 2016–2023* or *still open* using the model's urban cut-off (10% building cover).
3. Summer surface temperature (2023–25) is compared across the classes.

Caveat: each year is an independent prediction, so small year-to-year changes are partly noise. We use the 2016 → 2023 change and check the threshold.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from coolcairo import growth
from coolcairo.config import load_config
from coolcairo.openbuildings import YEARS
from coolcairo.pipeline import build_blocks, load_presence
cfg = load_config()
g = cfg['growth']
first, last = g['first_year'], g['last_year']
blocks, _, _ = build_blocks(cfg)
presence = {y: load_presence(cfg, y) for y in YEARS}  # First run streams ~1 min per year.

## Building footprint area per year (model area)

In [ ]:
shares = growth.attach(blocks, growth.block_built_share(presence, cfg, g['presence_min']))
area = growth.built_area_km2(shares, cfg.block_size)
area.plot(marker='o', title='Building footprint area, model area (km²)')
print(f"{first}: {area[first]:.1f} km²  ->  {last}: {area[last]:.1f} km²  ({100 * (area[last] / area[first] - 1):+.1f}%)")

## Land-use transitions per block

In [ ]:
shares['transition'] = growth.transitions(shares, first, last, g['built_block_min'])
codes = growth.growth_class(shares, first, last, g['built_block_min'], g['denser_min'])
print(shares.transition.value_counts())
grid = codes.to_numpy().reshape(shares.row.max() + 1, shares.col.max() + 1)
from matplotlib.colors import ListedColormap
plt.figure(figsize=(7, 7))
plt.imshow(grid, cmap=ListedColormap(['#a08f6c', '#737a8a', '#f2c14e', '#eb6834']), vmin=-0.5, vmax=3.5)
plt.title('Open / built before 2016 / denser / new since 2016'); plt.axis('off')

## Is new development hot?
Summer LST by transition class, and residents (WorldPop 2024) in newly built blocks.

In [ ]:
print(growth.heat_by_transition(shares).round(2))
new = shares[shares.transition == growth.NEWLY_BUILT]
print(f"Residents in newly built blocks: {new.population.clip(lower=0).sum():,.0f}")
print(f"Newly built blocks with < 5% OSM footprint cover: {(new.roof_frac < 0.05).mean():.0%}")

## Sensitivity to the presence cut-off

In [ ]:
rows = {}
for t in (0.4, 0.5, 0.6):
    s = growth.attach(blocks, growth.block_built_share(presence, cfg, t))
    a = growth.built_area_km2(s, cfg.block_size)
    n = (growth.transitions(s, first, last, g['built_block_min']) == growth.NEWLY_BUILT).sum()
    rows[t] = {'area_first_km2': a[first], 'area_last_km2': a[last], 'newly_built_blocks': n}
pd.DataFrame(rows).T.round(2)